# Task A + B: private execution launcher
One shared reviewed bundle; Python 3.12; select an NVIDIA T4 or better GPU.
Only synthetic engineering runs execute here. Real training never starts automatically.
Save a private copy and disable output saving. Supply the ZIP and independently retained
SHA-256 in the first code cell. Select a, b or both below. Private runs have separate
Task A/B directories. Restart session: rerun from top; no manual CODE_ROOT repair.
A PASS in CPU synthetic mode is never CUDA evidence. GPU mode fails without CUDA.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive').resolve()
BUNDLE_PATH = DRIVE_BASE / 'reviewed_code/task-ab-code.zip'  # Set your code-only ZIP.
BUNDLE_SHA256 = ''  # Required SHA-256 recorded independently when building the ZIP.
CONFIG_PATH = None  # Later: Path to your PRIVATE JSON config; never print its contents.


In [ ]:
import hashlib
import json
import stat
import zipfile

if (len(BUNDLE_SHA256) != 64 or any(c not in '0123456789abcdef' for c in BUNDLE_SHA256)
        or not BUNDLE_PATH.resolve().is_relative_to(DRIVE_BASE)
        or not BUNDLE_PATH.is_file() or BUNDLE_PATH.stat().st_size > 20 * 1024 * 1024):
    raise RuntimeError('Configure the private reviewed code bundle and independent SHA-256.')
blob = BUNDLE_PATH.read_bytes()
if hashlib.sha256(blob).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError('Code bundle checksum mismatch.')
CODE_ROOT = Path('/content') / ('task_ab_code_' + BUNDLE_SHA256[:16])
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    entries = archive.infolist()
    names = [entry.filename for entry in entries]
    if (len(entries) > 64 or len(set(names)) != len(names)
            or sum(entry.file_size for entry in entries) > 20 * 1024 * 1024):
        raise RuntimeError('Code bundle resource or duplicate-entry check failed.')
    for entry in entries:
        name = Path(entry.filename)
        if (name.is_absolute() or '..' in name.parts or '\\' in entry.filename
                or stat.S_ISLNK(entry.external_attr >> 16) or entry.file_size > 2 * 1024 * 1024):
            raise RuntimeError('Unsafe code bundle member.')
    manifest = json.loads(archive.read('code_manifest.json'))
    if (manifest.get('code_only') is not True or manifest.get('schema_version') != 1
            or set(names) != set(manifest['files']) | {'code_manifest.json'}):
        raise RuntimeError('Code-only allowlist check failed.')
    # Persist the verified manifest too; restart cells need it on disk.
    verified_files = {**manifest['files'], 'code_manifest.json':
                      hashlib.sha256(archive.read('code_manifest.json')).hexdigest()}
    for name, expected in verified_files.items():
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != expected:
            raise RuntimeError('Bundled source checksum mismatch.')
        target = CODE_ROOT / name
        if any(parent.is_symlink() for parent in (target, *target.parents)):
            raise RuntimeError('Code destination symlink rejected.')
        if target.exists():
            if target.read_bytes() != content:
                raise RuntimeError('Existing code snapshot differs; use a fresh runtime.')
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
print({'reviewed_code_verified': True})

STATE_PATH = Path('/content/brainagenticthon-code-state.json')
STATE_PATH.write_text(json.dumps({'code_root': str(CODE_ROOT), 'bundle_sha256': BUNDLE_SHA256,
    'manifest_sha256': hashlib.sha256(
        (CODE_ROOT / 'code_manifest.json').read_bytes()).hexdigest()}))


In [ ]:
# ruff: noqa: E402
import hashlib
import json
import sys
from pathlib import Path

# Survives Restart session: recover and revalidate the reviewed snapshot.
state = json.loads(Path('/content/brainagenticthon-code-state.json').read_text())
CODE_ROOT = Path(state['code_root'])
manifest_blob = (CODE_ROOT / 'code_manifest.json').read_bytes()
if hashlib.sha256(manifest_blob).hexdigest() != state['manifest_sha256']:
    raise RuntimeError('Reviewed manifest changed; rerun bundle verification.')
manifest = json.loads(manifest_blob)
for name, expected in manifest['files'].items():
    target = CODE_ROOT / name
    if (Path(name).is_absolute() or '..' in Path(name).parts
            or target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
            or hashlib.sha256(target.read_bytes()).hexdigest() != expected):
        raise RuntimeError('Reviewed snapshot changed; rerun bundle verification.')
import subprocess

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('This pinned research environment requires Python 3.12.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r',
                str(CODE_ROOT / 'requirements-task-a.txt')], check=True)
sys.path.insert(0, str(CODE_ROOT / 'src'))

# Check this research profile; Colab may contain unrelated incompatible packages.
from importlib.metadata import version

for requirement in (CODE_ROOT / 'requirements-task-a.txt').read_text().splitlines():
    if requirement.strip() and not requirement.startswith('#'):
        package, expected = requirement.strip().split('==')
        if version(package).split('+')[0] != expected:
            raise RuntimeError('Installed research dependency differs from reviewed pin.')


In [ ]:
# ruff: noqa: E402
import hashlib
import json
import sys
from pathlib import Path

# Survives Restart session: recover and revalidate the reviewed snapshot.
state = json.loads(Path('/content/brainagenticthon-code-state.json').read_text())
CODE_ROOT = Path(state['code_root'])
manifest_blob = (CODE_ROOT / 'code_manifest.json').read_bytes()
if hashlib.sha256(manifest_blob).hexdigest() != state['manifest_sha256']:
    raise RuntimeError('Reviewed manifest changed; rerun bundle verification.')
manifest = json.loads(manifest_blob)
for name, expected in manifest['files'].items():
    target = CODE_ROOT / name
    if (Path(name).is_absolute() or '..' in Path(name).parts
            or target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
            or hashlib.sha256(target.read_bytes()).hexdigest() != expected):
        raise RuntimeError('Reviewed snapshot changed; rerun bundle verification.')
sys.path.insert(0, str(CODE_ROOT / 'src'))
import nibabel as nib
import numpy as np
import torch

import brainagenticthon

if not Path(brainagenticthon.__file__).resolve().is_relative_to(CODE_ROOT):
    raise RuntimeError('Previously imported project differs from reviewed bundle; restart runtime.')
if (torch.__version__.split('+')[0] != '2.6.0'
        or np.__version__ != '2.5.4' or nib.__version__ != '5.4.2'):
    raise RuntimeError('Runtime imports differ from pinned packages; restart the runtime.')
print({'dependencies_verified': True, 'cuda_available': torch.cuda.is_available()})


In [ ]:
import uuid
from datetime import datetime, timezone

TASKS = 'both'  # 'a', 'b', or 'both'
MODE = 'synthetic_gpu'  # 'synthetic' explicitly selects CPU; real modes are refused.
PRIVATE_ROOT = DRIVE_BASE / 'BrainAgenticthon-private'
RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUTPUT = PRIVATE_ROOT / 'execution_runs' / RUN_NAME
# Optional existing private config paths for read-only readiness inspection, never real training.
READINESS_CONFIG_A = None
READINESS_CONFIG_B = None


In [ ]:
from brainagenticthon.execution.validation import run_validation

REPORT = run_validation(OUTPUT, tasks=TASKS, mode=MODE)
print({'status': REPORT['status'], 'gpu_executed': REPORT['gpu_executed'],
       'tasks': {key: value['status'] for key, value in REPORT['tasks'].items()},
       'report_file': 'execution_report.json', 'clinical_approval': False})


In [ ]:
from brainagenticthon.execution.readiness import save_inspection

for task, config_path in (('a', READINESS_CONFIG_A), ('b', READINESS_CONFIG_B)):
    if TASKS in {task, 'both'}:
        if (config_path is not None
                and not Path(config_path).resolve().is_relative_to(DRIVE_BASE)):
            raise RuntimeError('Readiness configuration must remain on authorized private Drive.')
        readiness = save_inspection(
            config_path, OUTPUT / ('readiness_' + task + '.json'), task=task)
        print({'task': task, 'readiness_status': readiness['status'], 'training_started': False})


Private execution_report.json and its SHA-256 record actual device/packages, losses,
AMP, model updates, checkpoint integrity, trusted reload/resume and synthetic held-out
metrics. Task A and B checkpoints stay under task_a/ and task_b/. Rerunning creates a new
run; existing outputs are never overwritten. CUDA unavailable produces machine-readable
FAIL; inspect that report and select a GPU runtime rather than overriding the device.
Readiness reports contain required evidence and aggregate gate results only. They do
not approve diagnoses or change eligibility. Use notebooks 05/06 separately for real
training only after independent approvals and all gates pass. Task A localization and
BraTS2021-to-existing-registry output export remain explicitly unsupported. No Task C
integration or clinical readiness is claimed. See docs/task_ab_execution_delivery.md.
